# Prompt 26: Debugging Spark Applications
## Databricks Certified Associate Developer for Apache Spark
### Topic 4 — Troubleshooting and Tuning Spark Applications (10%)

---

## Reading the Spark UI

### How to Access the Spark UI
| Context | URL |
|---------|-----|
| Local / standalone | `http://localhost:4040` (while application is running) |
| YARN cluster | Application master URL from YARN Resource Manager |
| Databricks | Cluster → Spark UI link in the cluster details page |
| History server (after app finishes) | `http://history-server:18080` — requires `spark.eventLog.enabled=true` |

### Tab-by-Tab Reference

| Tab | What it shows | Key debugging use |
|-----|---------------|------------------|
| **Jobs** | All jobs (one per action); status; duration; stage count | First stop — find the slow/failed job |
| **Stages** | Stages within a job; shuffle read/write bytes; skipped stages | Find expensive shuffle; skipped = cache hit |
| **Tasks** | Per-task metrics: duration, GC time, shuffle read/write, input bytes | Detect data skew (median vs max duration) |
| **SQL** | Graphical physical plan for DataFrame/SQL queries | Check join type; see predicate pushdown |
| **Storage** | Cached RDDs/DataFrames: storage level, fraction cached, memory used | Verify cache is being used |
| **Environment** | All configuration properties in effect | Confirm `spark.conf.set()` was applied |
| **Executors** | Executor list: memory used, GC time, tasks completed | Spot an executor with high GC time or failures |

### Navigation Path: Failure → Root Cause Error Message

```
Jobs tab
  └─ Find FAILED job (red) → click job link
       └─ Stages tab (filtered to this job)
            └─ Find FAILED stage (red) → click stage link
                 └─ Tasks tab (within failed stage)
                      └─ Sort by Status = FAILED
                           └─ Expand failed task row
                                └─ "Full Exception" or "Errors" column
                                     └─ READ the Python traceback / Java stack trace

Alternative: Tasks tab → "Logs" link on the executor → search for ERROR in stderr
```

---

## Interpreting explain() Output

### The explain() Modes

| Mode | Command | Shows |
|------|---------|-------|
| Simple (default) | `df.explain()` | Physical plan only |
| Extended | `df.explain('extended')` | Parsed → Analysed → Optimised → Physical |
| Formatted | `df.explain('formatted')` | Readable indented physical plan with node IDs |
| Cost | `df.explain('cost')` | Physical plan + row count / size estimates |
| Codegen | `df.explain('codegen')` | Generated JVM bytecode for each stage |

### What to Look For in the Physical Plan

| Plan node | Meaning | Action |
|-----------|---------|--------|
| `BroadcastHashJoin` | Small table was broadcast — efficient | Good |
| `SortMergeJoin` | Both sides shuffled + sorted — expensive | Consider broadcast hint |
| `BroadcastExchange` | Broadcast is happening | Good |
| `Exchange hashpartitioning` | Shuffle is happening | Check shuffle bytes in Stages tab |
| `Filter` above `FileScan` | Filter pushed down to data source — good | Good |
| `Filter` below `HashAggregate` | Filter NOT pushed down — reads all data first | Move filter earlier |
| `AdaptiveSparkPlan` | AQE is active — plan may change at runtime | Normal with AQE enabled |
| `ColumnarToRow` | Reading Parquet columnar data | Normal |
| `Project` | Column selection / `select()` | Normal |
| `HashAggregate` | Aggregation (partial then final) | Normal for `groupBy` |

### Common Slow-Query Signatures

**1. No predicate pushdown (filter too late):**
```
Physical Plan:
+- Filter (event_type = click)              ← filter AFTER reading all data
   +- FileScan parquet [all columns]        ← reads ALL rows
```
Fix: call `filter()` before any join or aggregation so it appears above `FileScan`.

**2. SortMergeJoin when BroadcastHashJoin expected:**
```
Physical Plan:
+- SortMergeJoin [customer_id = customer_id]   ← both sides shuffled
   :- Sort [customer_id ASC]
   :  +- Exchange hashpartitioning(customer_id)
   +- Sort [customer_id ASC]
      +- Exchange hashpartitioning(customer_id)
```
Fix: `broadcast()` hint or increase `spark.sql.autoBroadcastJoinThreshold`.

---

## Logging in PySpark Applications

```python
import logging

# Configure Python logger in driver
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)s: %(message)s'
)
logger = logging.getLogger('my_pipeline')
logger.info('Pipeline started')

# Set Spark/JVM log level (controls what you see from Spark internals)
spark.sparkContext.setLogLevel('WARN')  # options: ALL, DEBUG, INFO, WARN, ERROR, FATAL, OFF

# Reduce noise — use WARN in production, DEBUG only when diagnosing
```

**Log levels and their use:**
| Level | Use |
|-------|-----|
| `OFF` | No Spark logs at all |
| `ERROR` | Only errors — quietest useful level |
| `WARN` | Errors + warnings — recommended for production |
| `INFO` | Normal Spark operation — verbose but useful for learning |
| `DEBUG` | Very verbose — use only for specific debugging |

---

## Intermediate Inspection Techniques

Use these checkpoints at each step of a pipeline to find where data breaks:

```python
# 1. See schema — check column names, types, nullable
df.printSchema()

# 2. See sample rows — truncate=False shows full string values
df.show(5, truncate=False)

# 3. Count rows — detect accidental row loss
print(f'Row count: {df.count()}')

# 4. Check null counts per column
import pyspark.sql.functions as F
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

# 5. Describe statistics
df.describe().show()

# 6. Check distinct values of a suspicious column
df.select('status').distinct().show()

# 7. Count by group to spot unexpected distribution
df.groupBy('category').count().orderBy(F.col('count').desc()).show()
```

---

## Identifying Data Skew from Spark UI Task Metrics

**Where:** Spark UI → Jobs → (slow job) → Stages → (slow stage) → Tasks tab

**Key metrics to compare:**

| Metric | Normal | Skewed |
|--------|--------|--------|
| Duration median vs max | Max ≤ 2× median | Max >> median (e.g., 10× or 100×) |
| Shuffle Read Size median vs max | Balanced | One task reads 90%+ of shuffle data |
| Input Size median vs max | Balanced | One task processes 90%+ of input |

The Tasks tab summary row shows: **min / 25th / median / 75th / max** — large gap between 75th and max = skew.

---

## Debugging UDFs

UDFs run on executors — `print()` output goes to **executor stderr logs**, not the driver console.

```python
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Pattern 1: print statements (visible in executor logs, not driver)
@udf(StringType())
def debug_udf(s):
    print(f'[DEBUG] input value: {s!r}')   # goes to executor stderr
    return s.upper() if s else None

# Pattern 2: try/except to surface errors gracefully
@udf(StringType())
def safe_udf(s):
    try:
        if s is None:
            return None
        return s.strip().upper()
    except Exception as e:
        return f'ERROR:{type(e).__name__}:{str(e)}'  # return error info as string

# View print output: Spark UI → Executors tab → click executor → stderr link
# Or on local mode: visible in the terminal output
```

---

## Cache-and-Inspect Pattern

Cache a DataFrame after an expensive step, inspect it, then continue — avoids recomputing the expensive step during inspection:

```python
# Step 1: expensive transformation
df_enriched = df_raw \
    .join(df_reference, on='product_id', how='left') \
    .withColumn('revenue', col('quantity') * col('price'))

# Step 2: cache and materialise before inspecting
df_enriched.cache()
df_enriched.count()   # triggers materialisation — cached from here

# Step 3: inspect without rerunning the join
df_enriched.printSchema()
df_enriched.show(5, truncate=False)
df_enriched.describe('revenue').show()
null_count = df_enriched.filter(col('revenue').isNull()).count()
print(f'Rows with null revenue: {null_count}')

# Step 4: continue with transformations — still reads from cache
df_final = df_enriched.filter(col('revenue') > 0).groupBy('category').sum('revenue')
df_final.show()

# Step 5: free cache when done
df_enriched.unpersist()
```

In [ ]:
# Cell 1: Setup
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, broadcast, rand, lit, when, count, udf, upper, trim,
    to_date, current_date, avg, sum as _sum
)
from pyspark.sql.types import StringType, IntegerType, DoubleType, StructType, StructField
import pyspark.sql.functions as F
import logging

spark = SparkSession.builder \
    .appName('DebuggingSparkApps') \
    .master('local[2]') \
    .config('spark.sql.shuffle.partitions', '4') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .getOrCreate()

# Reduce Spark internal log noise
spark.sparkContext.setLogLevel('ERROR')

# Python logger for driver-side logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)s: %(message)s'
)
logger = logging.getLogger('debug_notebook')
logger.info('SparkSession ready')
print('Setup complete')

In [ ]:
# Cell 2: Interpreting explain() output — predicate pushdown and join types

# Create sample DataFrames
df_events = spark.range(0, 10_000) \
    .withColumn('event_type', when(col('id') % 3 == 0, 'click')
                               .when(col('id') % 3 == 1, 'view')
                               .otherwise('purchase')) \
    .withColumn('user_id', (col('id') % 200).cast(IntegerType())) \
    .withColumn('amount', (rand() * 100).cast(DoubleType()))

df_users = spark.createDataFrame(
    [(i, f'User_{i}', 'active' if i % 2 == 0 else 'inactive') for i in range(200)],
    ['user_id', 'name', 'status']
)

# -------------------------------------------------------
# 1. Simple explain — physical plan only
# -------------------------------------------------------
print('=== 1. Simple explain() — physical plan only ===')
df_events.filter(col('event_type') == 'click') \
         .groupBy('user_id').agg(_sum('amount').alias('total')) \
         .explain()

# -------------------------------------------------------
# 2. Formatted explain — readable indented plan
# -------------------------------------------------------
print('\n=== 2. explain("formatted") — indented, readable plan ===')
df_events.filter(col('event_type') == 'click') \
         .groupBy('user_id').agg(_sum('amount').alias('total')) \
         .explain('formatted')

# -------------------------------------------------------
# 3. Join plan — BroadcastHashJoin vs SortMergeJoin
# -------------------------------------------------------
print('\n=== 3. Join plan WITHOUT broadcast hint (may show SortMergeJoin) ===')
df_events.join(df_users, on='user_id', how='inner') \
         .explain()

print('\n=== 3b. Join plan WITH broadcast hint (must show BroadcastHashJoin) ===')
df_events.join(broadcast(df_users), on='user_id', how='inner') \
         .explain()

# -------------------------------------------------------
# 4. Cost explain — includes row count estimates
# -------------------------------------------------------
print('\n=== 4. explain("cost") — row count and size estimates ===')
df_events.groupBy('event_type').count().explain('cost')

In [ ]:
# Cell 3: Intermediate inspection techniques — step-by-step pipeline debugging

raw_data = [
    (1, 'alice', '2026-03-15', 'electronics', 299.99),
    (2, 'bob',   '2026-03-16', 'clothing',    45.00),
    (3, None,    '2026-03-17', 'electronics', None),
    (4, 'diana', None,         'food',        12.50),
    (5, 'eve',   '2026-03-18', None,          88.00),
    (6, 'frank', '2026-03-19', 'electronics', 450.00),
]
schema = StructType([
    StructField('order_id',   IntegerType(), True),
    StructField('customer',   StringType(),  True),
    StructField('order_date', StringType(),  True),
    StructField('category',   StringType(),  True),
    StructField('amount',     DoubleType(),  True),
])
df_raw = spark.createDataFrame(raw_data, schema)

# CHECKPOINT 1: after read — verify schema and sample rows
print('=== CHECKPOINT 1: After read ===')
df_raw.printSchema()
df_raw.show(truncate=False)
print(f'Row count: {df_raw.count()}')

# CHECKPOINT 2: check null distribution before cleaning
print('\n=== CHECKPOINT 2: Null counts per column ===')
df_raw.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_raw.columns
]).show()

# Apply cleaning
df_clean = df_raw \
    .withColumn('order_date', to_date(col('order_date'), 'yyyy-MM-dd')) \
    .fillna({'customer': 'unknown', 'category': 'other', 'amount': 0.0})

# CHECKPOINT 3: after cleaning — verify types changed and nulls handled
print('\n=== CHECKPOINT 3: After cleaning ===')
df_clean.printSchema()
df_clean.show(truncate=False)

# CHECKPOINT 4: after filter — verify row count dropped as expected
df_filtered = df_clean.filter(col('amount') > 0)
print(f'\n=== CHECKPOINT 4: After filter(amount > 0) ===')
print(f'Row count: {df_filtered.count()} (expected to drop 1 row where amount=0)')
df_filtered.show(truncate=False)

# CHECKPOINT 5: after aggregation — verify result shape
df_agg = df_filtered.groupBy('category').agg(
    count('order_id').alias('order_count'),
    _sum('amount').alias('total_revenue'),
    avg('amount').alias('avg_order')
)
print('\n=== CHECKPOINT 5: After aggregation ===')
df_agg.show(truncate=False)

In [ ]:
# Cell 4: Logging patterns and log level control

logger = logging.getLogger('pipeline')

def run_pipeline_with_logging(df_input):
    logger.info(f'Pipeline started. Input row count: {df_input.count()}')

    # Step 1
    df_step1 = df_input.filter(col('amount') > 0)
    count_step1 = df_step1.count()
    logger.info(f'After filter: {count_step1} rows')

    # Step 2
    df_step2 = df_step1.withColumn('revenue_band',
        when(col('amount') < 50, 'low')
        .when(col('amount') < 200, 'medium')
        .otherwise('high')
    )
    logger.info('Revenue band column added')

    # Step 3
    df_result = df_step2.groupBy('category', 'revenue_band') \
        .agg(count('order_id').alias('count'), _sum('amount').alias('total'))
    count_result = df_result.count()
    logger.info(f'Pipeline complete. Output row count: {count_result}')

    return df_result

result = run_pipeline_with_logging(df_clean)
result.show()

# -------------------------------------------------------
# Log level control for Spark internals
# -------------------------------------------------------
print('\n=== Spark log level control ===')
print('Current Spark log level: change with spark.sparkContext.setLogLevel(level)')
print('  Options: ALL, DEBUG, INFO, WARN, ERROR, FATAL, OFF')
print('  Recommended: ERROR for production, WARN for development')
print()
print('To see Spark query execution details in logs:')
print('  spark.sparkContext.setLogLevel("INFO")')
print('  # Look for lines: SparkContext, DAGScheduler, TaskScheduler, BlockManager')
print()
print('To suppress all Spark log noise while debugging Python logic:')
print('  spark.sparkContext.setLogLevel("ERROR")')

In [ ]:
# Cell 5: Debugging UDFs — print statements and try/except patterns

df_test = spark.createDataFrame(
    [('  Alice  ',), ('bob',), (None,), ('  CHARLIE',), ('',)],
    ['name']
)

# -------------------------------------------------------
# Pattern 1: print statements inside UDF
# Note: in local mode, these appear in the terminal.
# On a cluster they appear in executor stderr logs.
# -------------------------------------------------------
print('=== UDF with print debugging ===')
@udf(StringType())
def debug_normalize(s):
    print(f'[UDF DEBUG] input: {s!r}, type: {type(s).__name__}')  # executor stderr
    if s is None or s.strip() == '':
        return None
    return s.strip().title()

df_test.withColumn('normalized', debug_normalize(col('name'))).show(truncate=False)

# -------------------------------------------------------
# Pattern 2: try/except — return error info as string so failures are visible in output
# -------------------------------------------------------
print('\n=== UDF with try/except for error surfacing ===')
@udf(StringType())
def safe_parse_int_as_str(s):
    try:
        if s is None:
            return None
        return str(int(s))  # will fail if s is not a numeric string
    except (ValueError, TypeError) as e:
        return f'ERROR:{s}:{type(e).__name__}'  # surface the error in the data

df_numbers = spark.createDataFrame(
    [('42',), ('abc',), (None,), ('100',), ('7.5',)],
    ['value']
)
df_numbers.withColumn('parsed', safe_parse_int_as_str(col('value'))).show()

# -------------------------------------------------------
# Pattern 3: cache() after expensive step, inspect before continuing
# -------------------------------------------------------
print('\n=== Cache-and-inspect pattern ===')
df_expensive = df_clean \
    .withColumn('amount_doubled', col('amount') * 2) \
    .withColumn('name_upper', upper(trim(col('customer'))))

# Cache after the expensive step
df_expensive.cache()
df_expensive.count()  # materialise — subsequent actions hit cache

# Inspect without rerunning the transformation
print('Schema after caching:')
df_expensive.printSchema()
print('Sample rows from cache:')
df_expensive.show(3, truncate=False)
print('Is cached:', df_expensive.is_cached)

# Continue with more transformations — reads from cache
df_final = df_expensive.filter(col('amount_doubled') > 50).groupBy('category').count()
df_final.show()

df_expensive.unpersist()
print('Unpersisted. is_cached:', df_expensive.is_cached)

spark.stop()
print('\nDone.')

## Real-World Scenarios

<details>
<summary>Scenario 1: Finding the root cause of a failed production job via Spark UI</summary>

**Situation:**
A nightly ETL job fails at 2 AM. The email alert says `SparkException: Job aborted due to stage failure`. There is no obvious error in the driver logs. The developer needs to find the actual cause.

**Navigation:**
```
1. History Server: http://history-server:18080
   → Find the application (by name + timestamp)
   → Click to open its Spark UI

2. Jobs tab
   → Find the FAILED job (red)
   → Note the job description (often the action name: count, write, etc.)

3. Stages tab (filtered to failed job)
   → Find the FAILED stage
   → Check: how much shuffle read/write? Which stage in the lineage?

4. Tasks tab (within the failed stage)
   → Filter: Status = FAILED
   → Expand the failed task
   → Read: Full Exception → PythonException traceback

Found: AttributeError: 'NoneType' object has no attribute 'split'
       in UDF at line: return value.split(',')[0]
```

**Fix:**
```python
@udf(StringType())
def safe_first_element(value):
    if value is None:
        return None
    parts = value.split(',')
    return parts[0] if parts else None
```

**Exam Sub-topic:** Spark UI Jobs/Stages/Tasks navigation; reading executor error messages; UDF null safety
</details>

<details>
<summary>Scenario 2: Using explain() to discover a missing predicate pushdown</summary>

**Situation:**
A query reads a large Parquet table (500GB) and filters for a single day's data. It takes 45 minutes despite having a `filter()` call. The developer expects Parquet partition pruning to kick in.

**Diagnosis:**
```python
df_orders.filter(col('order_date') == '2026-04-22') \
         .groupBy('region').agg(F.sum('amount')) \
         .explain('formatted')

# Output shows:
# +- HashAggregate
#    +- Filter (order_date = 2026-04-22)    ← filter AFTER scanning
#       +- FileScan parquet [all columns]   ← scans ALL 500GB
#          PartitionFilters: []             ← EMPTY — no pruning!
```

**Root cause:** The Parquet file was not written with `partitionBy('order_date')`, so Spark can't prune files.

**Fix (write-time):**
```python
df_orders.write.partitionBy('order_date').parquet('/data/orders_partitioned')
# Now: PartitionFilters: [(order_date = 2026-04-22)] — reads only 1 day's files
```

**Exam Sub-topic:** `explain('formatted')`; predicate pushdown; Parquet partition pruning
</details>

<details>
<summary>Scenario 3: Step-by-step pipeline debugging with printSchema() and show() to find where rows disappear</summary>

**Situation:**
A pipeline starts with 1 million rows and produces only 50,000 in the output — expected ~900,000. The developer needs to find which step is dropping rows.

**Debugging approach:**
```python
df_raw = spark.read.parquet('/data/events')
print(f'Step 0 — raw:        {df_raw.count():,} rows')       # 1,000,000

df_s1 = df_raw.filter(col('event_type').isin(['click', 'view', 'purchase']))
print(f'Step 1 — event type: {df_s1.count():,} rows')         # 980,000 — ok

df_s2 = df_s1.join(df_users, on='user_id', how='inner')
print(f'Step 2 — join users: {df_s2.count():,} rows')          # 52,000 — !! 928k dropped!
# Most user_ids in events don't exist in df_users

# Fix: use left join to keep all events
df_s2_fixed = df_s1.join(df_users, on='user_id', how='left')
print(f'Step 2 fixed — left: {df_s2_fixed.count():,} rows')    # 980,000 — ok

# Also: check null distribution to understand the data
df_s2_fixed.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_s2_fixed.columns
]).show()
```

**Exam Sub-topic:** Intermediate `count()` checkpoints; inner vs left join; diagnosing unexpected row loss
</details>

<details>
<summary>Scenario 4: Using the cache-and-inspect pattern to find null values after a complex join</summary>

**Situation:**
After a chain of joins and transformations, the final aggregation produces unexpected `null` totals. The developer needs to find where nulls were introduced without rerunning the expensive joins.

**Approach:**
```python
# Build up to the expensive step
df_intermediate = df_events \
    .join(df_users, on='user_id', how='left') \
    .join(df_products, on='product_id', how='left') \
    .withColumn('revenue', col('quantity') * col('unit_price'))

# Cache and inspect BEFORE the aggregation
df_intermediate.cache()
df_intermediate.count()  # materialise

# Check for nulls — found: unit_price is null for 15% of rows
df_intermediate.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in ['user_id', 'product_id', 'quantity', 'unit_price', 'revenue']
]).show()
# unit_price: 150,000 nulls — product_id missing from product reference table

# Fix: fill missing unit_price with 0 or investigate product catalog
df_fixed = df_intermediate.fillna({'unit_price': 0.0})
df_fixed.withColumn('revenue', col('quantity') * col('unit_price')) \
        .groupBy('category').agg(F.sum('revenue')).show()

df_intermediate.unpersist()
```

**Exam Sub-topic:** Cache-and-inspect pattern; null diagnosis; `count()` as a materialisation trigger; `unpersist()`
</details>

<details>
<summary>Scenario 5: Identifying data skew in Spark UI and correlating with a specific join key</summary>

**Situation:**
A join stage takes 2 hours. In the Spark UI Tasks tab: median task duration = 8 seconds, max task duration = 1 hour 58 minutes. One task is processing 60% of all shuffle data.

**Investigation code:**
```python
# Find the skewed key — look at key distribution
df_orders.groupBy('merchant_id') \
         .count() \
         .orderBy(col('count').desc()) \
         .show(10)

# Result:
# +-------------+---------+
# | merchant_id |   count |
# +-------------+---------+
# | MEGA_CORP   | 8500000 |  ← 60% of rows
# | ACME_INC    |  200000 |
# | ...

# Option 1: Enable AQE skew join (automatic)
spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'true')

# Option 2: Manual salting
SALT = 20
df_orders_salted = df_orders.withColumn(
    'salted_id',
    F.concat(col('merchant_id'), F.lit('_'), (F.rand() * SALT).cast('int').cast('string'))
)
df_merchants_replicated = df_merchants \
    .crossJoin(spark.range(SALT).withColumnRenamed('id', 's')) \
    .withColumn('salted_id', F.concat(col('merchant_id'), F.lit('_'), col('s').cast('string')))

df_result = df_orders_salted.join(df_merchants_replicated, on='salted_id').drop('salted_id', 's')
```

**After fix:** Tasks tab shows balanced durations — max ≈ 2× median. Stage completes in 6 minutes.

**Exam Sub-topic:** Skew detection via Tasks tab; `groupBy().count()` to find skewed keys; AQE skew join; salting
</details>

## Exam Cheat Sheet

| Question | Answer |
|----------|--------|
| Spark UI default URL | `http://localhost:4040` (local); port varies on cluster |
| History server URL | `http://history-server:18080` (requires `spark.eventLog.enabled=true`) |
| Where to find the failed task error | Jobs → Stages → Tasks → expand failed task → "Full Exception" |
| Tab to verify a broadcast join fired | **SQL tab** — look for `BroadcastHashJoin` node |
| Tab to detect data skew | **Tasks** tab within a Stage — compare median vs max duration |
| Tab to verify config was applied | **Environment** tab |
| Tab to verify cache is being used | **Storage** tab |
| Tab to find executor with high GC | **Executors** tab — GC time column |
| `explain()` default mode | Physical plan only |
| Readable physical plan | `df.explain('formatted')` |
| All 4 plan stages | `df.explain('extended')` (parsed → analysed → optimised → physical) |
| With row count estimates | `df.explain('cost')` |
| Sign of no predicate pushdown | `Filter` below `FileScan` in explain output; `PartitionFilters: []` |
| Reduce Spark log noise | `spark.sparkContext.setLogLevel('WARN')` |
| Python logger in driver | `import logging; logger = logging.getLogger('name')` |
| Where UDF print() output goes | Executor stderr logs (not driver console); visible in UI → Executors → stderr |
| Best way to debug UDF errors | Wrap body in `try/except` and return error as string; add `None` guard |
| Cache-and-inspect pattern | `df.cache(); df.count(); df.show(); df.printSchema()` |
| Why `count()` after `cache()` | Triggers materialisation — fills cache so next actions are fast |
| `truncate=False` in `show()` | Shows full string values without truncation — useful for debugging |
| Check null counts | `df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()` |

---

## Practice Q&A

<details>
<summary>Q1: A developer runs df.explain() and sees SortMergeJoin instead of the expected BroadcastHashJoin. What are two possible causes and fixes?</summary>

**A:**
1. **The smaller DataFrame exceeds `autoBroadcastJoinThreshold` (default 10MB).** Spark decided it was too large to broadcast automatically.
   - Fix A: Increase the threshold: `spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '50MB')`
   - Fix B: Use an explicit broadcast hint: `large_df.join(broadcast(small_df), on='key')`

2. **Statistics are not available** and Spark can't estimate the size. With `spark.sql.adaptive.enabled=false`, Spark defaults to SortMergeJoin when it's uncertain.
   - Fix: Enable AQE (`spark.sql.adaptive.enabled=true`) — at runtime, AQE can switch to BroadcastHashJoin once it knows the actual size. Or use explicit `broadcast()` hint.
</details>

<details>
<summary>Q2: What is the explain() plan signature that tells you a filter was NOT pushed down to the data source?</summary>

**A:** In the physical plan, the `Filter` node appears **below** `HashAggregate` / `Join` — meaning the full dataset was scanned first and filtering happened afterward:

```
HashAggregate
+- Filter (event_date = 2026-04-22)     ← filter AFTER aggregation start
   +- FileScan parquet [all columns]    ← all data was read
      PartitionFilters: []              ← no partition pruning
```

A properly pushed-down filter looks like:
```
FileScan parquet
  PartitionFilters: [(event_date = 2026-04-22)]  ← pruning at source
  PushedFilters: [IsNotNull(amount)]              ← pushed into file scan
```
</details>

<details>
<summary>Q3: Where does print() output from inside a UDF appear, and how do you view it?</summary>

**A:** `print()` statements inside a UDF run on executors, not the driver. The output goes to **executor stderr logs**.

- **Local mode:** visible directly in the terminal/console running the Spark application
- **Cluster mode:** navigate to Spark UI → **Executors** tab → click the executor that ran the task → click the **stderr** log link
- **Databricks:** click the cluster → Spark UI → Executors → stderr

For easier debugging, use the `try/except` pattern in UDFs to return error information as a string column value — visible directly in `df.show()` output.
</details>

<details>
<summary>Q4: Why should you call df.count() immediately after df.cache(), and what does the Storage tab show afterward?</summary>

**A:** `cache()` is lazy — it marks the DataFrame to be cached when first computed, but doesn't actually compute or store anything yet. Calling `count()` (or any action) forces Spark to compute the DataFrame and fill the cache.

Without the `count()` call, the first subsequent action (e.g., `show()`) will compute the DataFrame and cache it simultaneously. The second action will then use the cache. The explicit `count()` after `cache()` ensures all subsequent actions — including the first one — benefit from the cache.

**Storage tab after materialisation:** Shows the cached DataFrame with:
- Storage level (e.g., `Disk Memory Deserialized 1x Replicated`)
- Fraction cached (0–100%)
- Memory used
- Disk used (if it spilled)
</details>

<details>
<summary>Q5: What two metrics in the Spark UI Tasks tab indicate data skew, and what is the preferred fix in Spark 3.2+?</summary>

**A:** The two key indicators of data skew in the Tasks tab:
1. **Max task duration >> median task duration** — e.g., median = 10s but max = 20min indicates one task processes far more data
2. **Max shuffle read size >> median shuffle read size** — one task receives a disproportionately large share of shuffle data

The summary row at the top of the Tasks table shows percentile statistics (min / 25th / median / 75th / max) for both duration and shuffle read. A large gap between 75th and max confirms skew.

**Preferred fix in Spark 3.2+:** Enable AQE skew join optimisation:
```python
spark.conf.set('spark.sql.adaptive.enabled', 'true')           # default True in 3.2+
spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'true')  # default True
```
AQE automatically detects and splits skewed sort-merge join partitions at runtime, spreading them across multiple tasks. For groupBy skew (not just joins), manual salting is still needed.
</details>